In [ ]:
import gc
import sys
from itertools import product
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image
import torch

from vision_unlearning.unlearner import UCE, ConceptType
from vision_unlearning.utils.logger import setup_loggers
from vision_unlearning.metrics import MetricImageTextSimilarity

In [ ]:
setup_loggers(modules_info=['vision_unlearning.'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_base_name = 'CompVis/stable-diffusion-v1-4'
output_dir = 'model'

# For unlearning
concept_forget = 'cat'
guide_concepts="animals"
preserve_concepts="lion; tiger; leopard"

# For evaluation
example_prompts_forget = [
    "Picture of a cat",
    "A cat playing",
    "A household feline pet ploting the global domination",
]
example_prompts_retain = [
    "Picture of a church",
    "church",
    "Photograph of a dog",
    "A concert from queen",
]

In [ ]:
!nvidia-smi

# Original model
As you can see, it can generate the undesired concept

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(model_base_name, torch_dtype=torch.float16, safety_checker=None).to(device)

In [ ]:
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()

In [ ]:
del pipeline
gc.collect()
torch.cuda.empty_cache()

# Hypeparameter tuning
Find ideal set of hypeparameters for this specific unlearning problem.
If you already know a good set of hyperparameters for your specific unlearning problem, **you can skip this step**.

We are doing a simple grid search of some hyperparam combinations. Maximize utility function.

Utility function = difference between forget and retain clip score, constrained to retain score > 26.
Remember: higher clip score = better, so it must be low for forget and high for retain.

This is **very simplified**, and many things are problemmatic: just 2 evaluation images (noisy and unrealible result), just one prompt template, reliying purely on clip, among others.

In [ ]:
erase_scales = [0.3, 0.5, 0.7, 0.9]
preserve_scales = [0.5, 1.0, 1.5, 2.5]
lambs = [0.3, 0.5, 0.8]

best_utility = -np.inf
best_erase_scale = None
best_preserve_scale = None
best_lamb = None

clip_metric = MetricImageTextSimilarity(metrics=['clip'])

for erase_scale, preserve_scale, lamb in product(erase_scales, preserve_scales, lambs):
    print(f"Trying combination of erase_scale={erase_scale}, preserve_scale={preserve_scale}, lamb={lamb}")
    unlearner = UCE(
        pretrained_model_name_or_path=model_base_name,
        erase_scale=erase_scale,
        preserve_scale=preserve_scale,
        lamb=lamb,
        edit_concepts=concept_forget,
        guide_concepts=guide_concepts,
        preserve_concepts=preserve_concepts,
        device=device,
        concept_type=ConceptType.Object,
        output_dir=output_dir,
    )
    unlearner.train()

    pipeline = AutoPipelineForText2Image.from_pretrained(output_dir, torch_dtype=torch.float16, safety_checker=None).to(device)
    image_forget = pipeline(example_prompts_forget[0]).images[0]
    image_retain = pipeline(example_prompts_retain[0]).images[0]
    clip_forget: float = clip_metric.score(image_forget, example_prompts_forget[0])["clip"]
    clip_retain: float = clip_metric.score(image_retain, example_prompts_retain[0])["clip"]

    plt.subplot(1, 2, 1)
    plt.imshow(image_forget)
    plt.title(f'{example_prompts_forget[0]} (forget concept)\nClip={clip_forget:.2f}')
    plt.subplot(1, 2, 2)
    plt.imshow(image_retain)
    plt.title(f'{example_prompts_retain[0]} (retain concept)\nClip={clip_retain:.2f}')
    plt.show()

    utility = clip_retain - clip_forget - (100 if clip_retain < 26 else 0)
    if utility > best_utility:
        best_utility = utility
        best_erase_scale = erase_scale
        best_preserve_scale = preserve_scale
        best_lamb = lamb

print(f"Best combination: erase_scale={erase_scale}, preserve_scale={preserve_scale}, lamb={lamb}")

# Unlearning
Performing the actual model update

In [ ]:
unlearner = UCE(
    pretrained_model_name_or_path=model_base_name,
    #erase_scale=0.4,
    #preserve_scale=1.0,
    #lamb=0.5,
    erase_scale=best_erase_scale,
    preserve_scale=best_preserve_scale,
    lamb=best_lamb,
    edit_concepts=concept_forget,
    guide_concepts=guide_concepts,
    preserve_concepts=preserve_concepts,
    device=device,
    concept_type=ConceptType.Object,
    output_dir=output_dir,
)

eval_results = unlearner.train()
pd.DataFrame([{'Name': r.metric_name, 'Value': r.metric_value} for r in eval_results])

In [ ]:
!du -h {output_dir}
!ls -lah {output_dir}

In [ ]:
del unlearner
gc.collect()
torch.cuda.empty_cache()

# Check unlearned model
as you can see, it does NOT generate the undesired concept anymore

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(output_dir, torch_dtype=torch.float16, safety_checker=None).to(device)
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()

In [ ]:
del pipeline
gc.collect()
torch.cuda.empty_cache()